In [ ]:
import sys
from datetime import timedelta

import polars as pl

sys.path.append("..")
from src.training.dataset import INPUT_FEATURES, TARGET

In [ ]:
lf = pl.scan_parquet("gs://mlops-solar-flux_offline_fs/")

In [ ]:
min_time = lf.select(pl.col("time").min()).collect().item()
max_time = lf.select(pl.col("time").max()).collect().item()

In [ ]:
min_time

In [ ]:
max_time

In [ ]:
duration = max_time - min_time
duration

In [ ]:
test_duration = duration * 0.2
test_duration

In [ ]:
test_start = (max_time - test_duration).replace(second=0, microsecond=0)
test_start

In [ ]:
lf.filter(pl.col("time") >= test_start).collect()

In [ ]:
train_end = test_start - timedelta(minutes=1440)
train_end

In [ ]:
train = lf.filter(pl.col("time") < train_end).collect()
train

In [ ]:
target = train.select(pl.col("max_24h_la"))
target

In [ ]:
X = train.select(
    pl.col(
        [
            "xrsb_flux",
            "lag_15",
            "lag_60",
            "lag_120",
            "lag_1440",
            "roll_max_720",
            "roll_std_720",
            "roll_mean_720",
            "deriv_1_5",
            "deriv_2_5",
            "roll_c_class_cross_720",
        ]
    )
)

In [ ]:
X[[0, 1, 4, 0]]

In [ ]:
min_time = train.select(pl.col("time").min()).item()
max_time = train.select(pl.col("time").max()).item()

In [ ]:
test_duration = (max_time - min_time) // 10
test_duration

In [ ]:
import xgboost as xgb

In [ ]:
d = xgb.DMatrix(X, label=target)

In [ ]:
d.get_label().std()

In [ ]:
d.get_label().mean()

In [ ]:
target.std().item()

In [ ]:
X.mean().to_dicts()[0]

In [ ]:
from datetime import datetime, timedelta

In [ ]:
min_time = lf.select(pl.col("time").min()).collect().item()
max_time = lf.select(pl.col("time").max()).collect().item()

test_duration = (max_time - min_time) * 0.2
test_start = (max_time - test_duration).replace(second=0, microsecond=0)
# Add 24h gap to avoid data leakage
train_end = test_start - timedelta(minutes=1440)

test_set = lf.filter(pl.col("time") >= test_start).collect()
train_set = lf.filter(pl.col("time") < train_end).collect()

In [ ]:
train_set.filter(pl.col("xrsb_flux") < 0)

In [ ]:
def ts_split(
    df: pl.DataFrame, max_time: datetime, test_start: datetime, gap: timedelta
) -> tuple[xgb.DMatrix, xgb.DMatrix]:
    print(max_time)
    print(test_start)
    train_end = test_start - gap
    print(train_end)

    val = df.filter((pl.col("time") >= test_start) & (pl.col("time") <= max_time))
    X_val = val.select(pl.col(INPUT_FEATURES))
    y_val = val.select(pl.col(TARGET))

    train = df.filter(pl.col("time") < train_end)
    X_train = train.select(pl.col(INPUT_FEATURES))
    y_train = train.select(pl.col(TARGET))

    return xgb.DMatrix(X_train, label=y_train), xgb.DMatrix(X_val, label=y_val)

In [ ]:
min_time = train_set.select(pl.col("time").min()).item()
max_time = train_set.select(pl.col("time").max()).item()

n_splits = 5
test_duration = (max_time - min_time) // (2 * n_splits)
gap = timedelta(minutes=1440)
current_max_time = max_time

In [ ]:
test_start = (current_max_time - test_duration).replace(second=0, microsecond=0)

dtrain, dval = ts_split(train_set, current_max_time, test_start, gap)
current_max_time = test_start - timedelta(minutes=1)

In [ ]:
dtrain.get_data()

In [ ]:
dtrain.get_label()

In [ ]:
dval.get_data()

In [ ]:
dval.get_label().std()

In [ ]:
dval.get_label().mean()

In [ ]:
evals_result = {}

bst = xgb.train(
    params={"objective": "reg:tweedie", "tweedie_variance_power": 1.1},
    dtrain=dtrain,
    evals=[(dtrain, "train"), (dval, "eval")],
    num_boost_round=2000,
    early_stopping_rounds=100,
    evals_result=evals_result,
)

In [ ]:
print(evals_result["eval"])

In [ ]:
y_train = dtrain.get_label()
y_val = dval.get_label()

train_mean, train_std = y_train.mean(), y_train.std()
val_mean, val_std = y_val.mean(), y_val.std()

train_rmse = evals_result["train"]["tweedie-nloglik@1.1"][bst.best_iteration]
val_rmse = evals_result["eval"]["tweedie-nloglik@1.1"][bst.best_iteration]

fold_metrics = {
    "train_rmse": train_rmse,
    "train_std": train_std,
    "train_mean": train_mean,
    "train_cvrmse": train_rmse / train_mean if train_mean != 0 else 0,
    "val_rmse": val_rmse,
    "val_std": val_std,
    "val_mean": val_mean,
    "val_cvrmse": val_rmse / val_mean if val_mean != 0 else 0,
}

In [ ]:
fold_metrics